# `qwen3-4b-4bit-qlora-s00-r0` — DGX Spark recipe

Container-aware. Run inside `nvcr.io/nvidia/pytorch:25.11-py3` (launched by `models/recipes/dgx_spark/launch.sh`).
Substrate-of-record: NVIDIA DGX Spark Unsloth playbook. Hardware notes: `.meta/hardware.md`.

**Decision log** (Leo, 2026-05-20):
- **Supersession-note rendering**: DEFERRED to r1. Diana's `_format.py` does not template-render `meta.gaap_supersession` into the assistant turn, AND the rendered split JSONL does not carry the supersession block in its slim `meta`. Re-rendering requires regenerating the splits. r0 trains on raw Spiceland gold; the drift is a post-hoc Vera eval column.
- **Loss weighting 0.85/0.15**: DEFERRED to r1. r0 trains uniformly (`SFTTrainer` default 1.0/1.0). Custom per-segment weighting requires a custom collator; not worth blocking r0.
- **r0 launch trigger**: user-gated. This notebook is *prepared*, not *executed*.

## §1 — Substrate guardrails

Fail fast if we are on the wrong host, wrong CUDA arch, wrong split version, or the seed split sha drifted from `eval/sft/splits/manifest.json`.

In [1]:
import hashlib, json, os, sys
from pathlib import Path

# Repo-root discovery: prefer the container mount `/workspace` if it exists
# (this notebook is designed to run inside nvcr.io/nvidia/pytorch:25.11-py3),
# otherwise walk up from cwd looking for a .git/ marker so host-side smoke
# tests work on Windows/macOS/Linux without a code change.
def _find_repo() -> Path:
    workspace = Path("/workspace")
    if workspace.is_dir() and (workspace / "eval" / "sft" / "splits").is_dir():
        return workspace
    forced = os.environ.get("REPO_ROOT")
    if forced:
        return Path(forced).resolve()
    p = Path.cwd().resolve()
    for d in [p, *p.parents]:
        if (d / ".git").is_dir() or (d / "eval" / "sft" / "splits").is_dir():
            return d
    raise RuntimeError(f"could not find repo root from {p}")

REPO = _find_repo()
print(f"REPO: {REPO}")
SEED_DIR = REPO / "eval/sft/splits/seed_00__351199285"
MANIFEST_PATH = REPO / "eval/sft/splits/manifest.json"
RUN_ID = "qwen3-4b-4bit-qlora-s00-r0"
RUN_DIR = REPO / f"models/runs/dgx_spark/{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

import torch
assert torch.cuda.is_available(), "CUDA not visible — launch via launch.sh with --gpus all"
cc = torch.cuda.get_device_capability(0)
name = torch.cuda.get_device_name(0)
print(f"device: {name}  compute_cap={cc[0]}.{cc[1]}")
assert cc == (12, 1), f"expected GB10 sm_120 (12,1), got {cc}"
assert "GB10" in name, f"expected GB10 device, got {name!r}"

for split in ("train", "valid", "test"):
    p = SEED_DIR / f"{split}.jsonl"
    assert p.exists(), f"missing {p}"

manifest = json.loads(MANIFEST_PATH.read_text())
assert manifest["source_jsonl_sha256"].startswith("fed6eb17de8be1e4"), (
    f"corpus drift: manifest source sha = {manifest['source_jsonl_sha256'][:16]}, "
    "expected spiceland9e-v1.1.0 = fed6eb17de8be1e4\u2026"
)
seed00 = next(s for s in manifest["seeds"] if s["seed_index"] == 0)
for split, want in seed00["file_sha256"].items():
    got = hashlib.sha256((SEED_DIR / f"{split}.jsonl").read_bytes()).hexdigest()
    assert got == want, f"{split} sha drift: got {got[:16]}\u2026 want {want[:16]}\u2026"
print("seed_00 splits sha-verified against manifest:")
for split, want in seed00["file_sha256"].items():
    print(f"  {split:<5} {want[:16]}\u2026")
print("corpus anchor: spiceland9e-v1.1.0")
print(f"counts: {seed00['split_counts']}")

REPO: /workspace
device: NVIDIA GB10  compute_cap=12.1
seed_00 splits sha-verified against manifest:
  train 5dce67f97fe300c1…
  valid e01a5ccbd2d3e27f…
  test  86e2914fb83d221c…
corpus anchor: spiceland9e-v1.1.0
counts: {'train': 3538, 'valid': 450, 'test': 464}


## §2 — Dataset + tokenizer

Diana's splits already carry rendered `messages` (system + user + assistant) per `eval/_format.py`. We map those to a single `text` field via the Qwen3 chat template. `meta.exclude_from_scoring` is already filtered upstream at split time (manifest reports 1 skip). The supersession-note rendering is **NOT** present in the split JSONL — see decision log above; r0 trains on raw Spiceland gold.

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer

BASE_MODEL_PRIMARY = "unsloth/Qwen3-4B-Instruct-2507"
BASE_MODEL_FALLBACK = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LEN = 2048

tok_probe = AutoTokenizer.from_pretrained(BASE_MODEL_PRIMARY, trust_remote_code=False)
TOKENIZER_REVISION = getattr(tok_probe, "name_or_path", BASE_MODEL_PRIMARY)
print(f"tokenizer probe ok: {TOKENIZER_REVISION}")
print(f"chat template present: {bool(tok_probe.chat_template)}")

ds = load_dataset(
    "json",
    data_files={
        "train": str(SEED_DIR / "train.jsonl"),
        "valid": str(SEED_DIR / "valid.jsonl"),
        "test":  str(SEED_DIR / "test.jsonl"),
    },
)
print({k: len(v) for k, v in ds.items()})

def render_text(rec):
    # rec["messages"] = [{role, content}, ...]; apply Qwen3 chat template with
    # the assistant turn fully present so train_on_responses_only can mask the
    # prompt half. Do not add a generation prompt.
    return tok_probe.apply_chat_template(
        rec["messages"], tokenize=False, add_generation_prompt=False
    )

ds = ds.map(lambda r: {"text": render_text(r)}, num_proc=4)
print("sample (first 600 chars of train[0].text):")
print(ds["train"][0]["text"][:600])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:277: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


tokenizer probe ok: unsloth/Qwen3-4B-Instruct-2507
chat template present: True
{'train': 3538, 'valid': 450, 'test': 464}
sample (first 600 chars of train[0].text):
<|im_start|>system
You are a graduate-level intermediate financial accounting tutor with deep knowledge of US GAAP (FASB ASC) and the Spiceland 9e curriculum. When you answer, cite the controlling chapter and learning objective if given, show your reasoning step by step for computational problems, and present journal entries in standard debit-on-top format. Use exact-decimal arithmetic for money amounts; do not round prematurely. If a problem provides tables, preserve and reference them in your reasoning.<|im_end|>
<|im_start|>user
[Chapter 1 · LO 01-01 · Environment of financial accounting an


## §3 — Base model (4-bit QLoRA via Unsloth `FastModel`)

Playbook-validated path: `FastModel.from_pretrained(load_in_4bit=True, full_finetuning=False)`. Try the Unsloth pre-quantized repo first; on miss, fall back to the official Qwen repo and let Unsloth auto-quantize.

In [3]:
from unsloth import FastModel, FastLanguageModel

BASE_MODEL_USED = BASE_MODEL_PRIMARY
try:
    model, tokenizer = FastModel.from_pretrained(
        model_name=BASE_MODEL_PRIMARY,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        load_in_8bit=False,
        full_finetuning=False,
    )
except Exception as e:
    print(f"primary load failed ({type(e).__name__}: {e!s:.200}); falling back to {BASE_MODEL_FALLBACK}")
    BASE_MODEL_USED = BASE_MODEL_FALLBACK
    model, tokenizer = FastModel.from_pretrained(
        model_name=BASE_MODEL_FALLBACK,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        load_in_8bit=False,
        full_finetuning=False,
    )
TOKENIZER_REVISION = getattr(tokenizer, "name_or_path", BASE_MODEL_USED)
print(f"base loaded: {BASE_MODEL_USED}")
print(f"tokenizer:   {TOKENIZER_REVISION}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.5: Fast Qwen3 patching. Transformers: 5.9.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.634 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
base loaded: unsloth/Qwen3-4B-Instruct-2507
tokenizer:   unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit


## §4 — LoRA adapter

Rank 16 chosen over the default 8 because the corpus has structured-output content (journal entries, multi-step rationales) that benefits from extra adapter capacity. `lora_alpha=32` keeps the alpha/r ratio at 2.0 (Unsloth's common ratio for QLoRA SFT); the playbook reference uses 16/16 but on small-corpus instruct tuning 2:1 is more stable.

In [4]:
SEED = 351199285
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=TARGET_MODULES,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

trainable params: 33,030,144 / 2,539,650,560 (1.30%)


## §5 — SFTTrainer config

Half-epoch eval (`eval_steps=110`): with 3,538 train rows / (batch 2 × grad-accum 4) ≈ 442 steps/epoch × 3 ≈ 1,326 total steps; 110 ≈ quarter-epoch, so we get ≈12 dev-loss readings to feed EarlyStoppingCallback (patience=3).

In [5]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

OUTPUT_DIR = str(RUN_DIR)

sft_cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.0,
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    dataset_text_field="text",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=110,
    save_strategy="steps",
    save_steps=110,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    fp16=False,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["valid"],
    args=sft_cfg,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/3538 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/450 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


## §6 — `train_on_responses_only` masking

Mask the prompt half so the loss is computed only on the assistant turn. Qwen3 instruction templates are `<|im_start|>user` ... `<|im_start|>assistant` ... `<|im_end|>`.

In [6]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
print("responses-only masking applied (Qwen3 im_start/im_end tags)")

Map (num_proc=24):   0%|          | 0/3538 [00:00<?, ? examples/s]

Filter (num_proc=24):   0%|          | 0/3538 [00:00<?, ? examples/s]

Unsloth: Removed 1 out of 3538 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


Map (num_proc=24):   0%|          | 0/450 [00:00<?, ? examples/s]

Filter (num_proc=24):   0%|          | 0/450 [00:00<?, ? examples/s]

responses-only masking applied (Qwen3 im_start/im_end tags)


## §7 — Train

Smoke-test projection (Phi-3.5-mini @ 4.30 samples/s on this host): Qwen3-4B at ≈ same throughput → 3,538 train rows × 3 epochs / 4.3 ≈ 41 min wall-clock for r0.

In [7]:
import time
t0 = time.time()
train_result = trainer.train()
wall_clock_s = time.time() - t0
print(f"train done in {wall_clock_s/60:.1f} min")
print(train_result.metrics)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"adapter + tokenizer saved \u2192 {OUTPUT_DIR}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,537 | Num Epochs = 3 | Total steps = 1,329
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
110,0.641248,0.742228
220,0.821764,0.701737
330,0.534466,0.688416
440,0.563885,0.679542
550,0.385064,0.695523
660,0.398725,0.684757
770,0.373408,0.677216
880,0.453335,0.671738
990,0.237162,0.707421
1100,0.372617,0.707571


[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/models/runs/qwen3-4b-4bit-qlora-s00-r0/checkpoint-110/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/models/runs/qwen3-4b-4bit-qlora-s00-r0/checkpoint-220/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/models/runs/qwen3-4b-4bit-qlora-s00-r0/checkpoint-330/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/models/runs/qwen3-4b-4bit-qlora-s00-r0/checkpoint-440/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/models/runs/qwen3-4b-4bit-qlora-s00-r0/checkpoint-550/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/models/runs/qwen3-4b-4bit-qlora-s00-r0/checkpoint-660/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace

train done in 91.5 min
{'train_runtime': 5489.149, 'train_samples_per_second': 1.933, 'train_steps_per_second': 0.242, 'total_flos': 1.2330875805023232e+17, 'train_loss': 0.5251932335293983, 'epoch': 2.7326172979084227}


[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/models/runs/qwen3-4b-4bit-qlora-s00-r0/tokenizer_config.json.


adapter + tokenizer saved → /workspace/models/runs/qwen3-4b-4bit-qlora-s00-r0


## §8 — Manifest emission

Per Leo's standing rule — every run pinned. The `source_jsonl_sha256` binds this adapter to `spiceland9e-v1.1.0`.

In [8]:
import datetime as dt

def file_sha256(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

adapter_path = RUN_DIR / "adapter_model.safetensors"
if not adapter_path.exists():
    candidates = list(RUN_DIR.glob("adapter_model*"))
    adapter_path = candidates[0] if candidates else None
adapter_sha = file_sha256(adapter_path) if adapter_path and adapter_path.exists() else None

best_dev_loss = None
log_history = getattr(trainer.state, "log_history", [])
for entry in log_history:
    if "eval_loss" in entry:
        v = entry["eval_loss"]
        if best_dev_loss is None or v < best_dev_loss:
            best_dev_loss = v

train_samples = len(ds["train"])
throughput = (train_result.metrics.get("train_samples_per_second")
              if train_result and train_result.metrics else None)

manifest_out = {
    "run_id": RUN_ID,
    "created_at": dt.datetime.utcnow().isoformat() + "Z",
    "base_model": BASE_MODEL_USED,
    "tokenizer_revision": TOKENIZER_REVISION,
    "adapter_sha256": adapter_sha,
    "source_jsonl_sha256": manifest["source_jsonl_sha256"],
    "corpus_anchor": "spiceland9e-v1.1.0",
    "train_split_sha256": seed00["file_sha256"]["train"],
    "valid_split_sha256": seed00["file_sha256"]["valid"],
    "holdout_split_sha256": seed00["file_sha256"]["test"],
    "seed_int": SEED,
    "recipe": {
        "method": "qlora-4bit",
        "rank": 16,
        "alpha": 32,
        "dropout": 0,
        "target_modules": TARGET_MODULES,
        "max_seq_length": MAX_SEQ_LEN,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-4,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.05,
        "num_train_epochs": 3,
        "optim": "adamw_8bit",
        "bf16": True,
        "loss_weighting": "uniform",
        "loss_weighting_note": (
            "r0 trains uniformly. Pat's 0.85/0.15 gold/supersession weighting deferred to r1; "
            "requires custom collator + supersession block plumbed into split records."
        ),
        "supersession_render": "absent",
        "supersession_note": (
            "r0 trains on raw Spiceland gold. eval/_format.py does not render "
            "meta.gaap_supersession into the assistant turn, and Diana's split JSONL drops "
            "the field. Deferred to r1."
        ),
        "responses_only_masking": True,
    },
    "final_dev_loss": best_dev_loss,
    "final_train_loss": train_result.metrics.get("train_loss") if train_result else None,
    "wall_clock_seconds": wall_clock_s,
    "throughput_samples_per_second": throughput,
    "train_samples": train_samples,
    "gpu_device": name,
    "cuda_compute_cap": f"{cc[0]}.{cc[1]}",
    "container_image": "nvcr.io/nvidia/pytorch:25.11-py3",
    "unsloth_version": "2026.5.5",
    "trl_version": "0.26.1",
    "datasets_version": "4.3.0",
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest_out, indent=2) + "\n")
print(json.dumps(manifest_out, indent=2))

{
  "run_id": "qwen3-4b-4bit-qlora-s00-r0",
  "created_at": "2026-05-21T17:03:42.113923Z",
  "base_model": "unsloth/Qwen3-4B-Instruct-2507",
  "tokenizer_revision": "unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit",
  "adapter_sha256": "a83c7a89975abac0a2a8d2227bb93c3bc12c636f3e9392972b26f3099f5b72cc",
  "source_jsonl_sha256": "fed6eb17de8be1e493b1a54277bdb70175502e517ea4cd4e6877914f437d213b",
  "corpus_anchor": "spiceland9e-v1.1.0",
  "train_split_sha256": "5dce67f97fe300c141c2c637af1cbfe6503b2063e3339bf0939dd741f2b30462",
  "valid_split_sha256": "e01a5ccbd2d3e27f36f6b4b54d2e863f127eaa9184498c129a6f2bc6c22b509a",
  "holdout_split_sha256": "86e2914fb83d221c86ef3db745a31164d756b7d0525ec76051cf6ef92b0857e7",
  "seed_int": 351199285,
  "recipe": {
    "method": "qlora-4bit",
    "rank": 16,
    "alpha": 32,
    "dropout": 0,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "max_seq_leng

/tmp/ipykernel_1323/1053945384.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": dt.datetime.utcnow().isoformat() + "Z",


## §9 — Quick eval (50-sample probe)

NOT production eval. Vera's `eval/run.py` is the production path (pending). This is a sanity-check that the adapter generates sensible completions before the user gates a full Vera run.

In [9]:
import random
FastLanguageModel.for_inference(model)

rng = random.Random(SEED)
test_rows = list(ds["test"])
probe = rng.sample(test_rows, k=min(50, len(test_rows)))
probe_path = RUN_DIR / "probe_predictions.jsonl"

with probe_path.open("w") as fh:
    for i, rec in enumerate(probe):
        msgs = rec["messages"][:2]
        prompt_text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
        completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        row = {
            "id": rec["meta"]["id"],
            "chapter": rec["meta"]["chapter"],
            "type": rec["meta"]["type"],
            "primary_bloom": rec["meta"]["primary_bloom"],
            "gold_assistant": rec["messages"][2]["content"],
            "prediction": completion,
        }
        fh.write(json.dumps(row) + "\n")
        if i < 3:
            print(f"--- probe {i} [{row['id']}, {row['type']}] ---")
            print(f"GOLD : {row['gold_assistant'][:200]}")
            print(f"PRED : {row['prediction'][:200]}")
print(f"probe predictions \u2192 {probe_path}")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- probe 0 [ch05_q0128_mc, Multiple Choice] ---
GOLD : **C**

A long delay before uncertainty resolves is an indicator that the constraint applies.
PRED : 


[transformers] Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- probe 1 [ch06_q0019_mc, Multiple Choice] ---
GOLD : **D**

PV = $500,000 × 0.66112* = $330,560
*PV of $1: n = 14; i = 3%
PRED : 


[transformers] Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- probe 2 [ch11_q0192_es, Essay/Problem] ---
GOLD : Answer:

| | Column 1 | Column 2 | Column 3 |
|:---|:---|:---|:---|
| Acquisition Date | 1/1/2016 | 1/1/2016 | 6/30/2016 |
| Cost | $250,000 | $320,000 | $320,000 |
| Accumulated Depreciation, 12/31/2
PRED : 


[transformers] Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https:/

probe predictions → /workspace/models/runs/qwen3-4b-4bit-qlora-s00-r0/probe_predictions.jsonl


## §10 — GGUF export (optional, gated)

For Ollama / llama.cpp portability. Disabled by default; flip the flag if a portable artifact is wanted post-Vera-PASS.

In [11]:
do_gguf_export = True

if do_gguf_export:
    gguf_dir = RUN_DIR / "gguf"
    gguf_dir.mkdir(exist_ok=True)
    model.save_pretrained_gguf(
        str(gguf_dir),
        tokenizer,
        quantization_method="q4_k_m",
    )
    print(f"GGUF q4_k_m \u2192 {gguf_dir}")
else:
    print("GGUF export skipped (flag off). Enable by setting do_gguf_export=True.")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/models/runs/qwen3-4b-4bit-qlora-s00-r0/gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /workspace/.hf_cache/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [08:43<00:00, 261.78s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:57<00:00, 28.89s/it]


Unsloth: Merge process complete. Saved to `/workspace/models/runs/qwen3-4b-4bit-qlora-s00-r0/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/workspace/models/runs/qwen3-4b-4bit-qlora-s00-r0/gguf_gguf/qwen3-4b-instruct-2507.BF16.gguf']
Unsloth: [2] C